# M0 pretrained smoke test
Chạy Qwen2.5-VL-3B-Instruct trên 20 mẫu validation cố định. Kết quả được lưu thẳng về Drive; không sử dụng test split để chỉnh prompt.

In [ ]:
import torch
from google.colab import drive

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > GPU'
!nvidia-smi

In [ ]:
import hashlib
import shutil
import zipfile
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')
SOURCE = DRIVE_ROOT / 'source/project_source.zip'
ARCHIVE = DRIVE_ROOT / 'datasets/images/coco_smoke_500.zip'
ANNOTATIONS = DRIVE_ROOT / 'datasets/processed/smoke_500.jsonl'
WORK_ROOT = Path('/content/vqa_work/coco')
REPO_ROOT = Path('/content/Vietnamese-VQA-Explanation')
EXPECTED_DATA_SHA256 = '4e0fce747dee9a1beaf36e2617deae6582115020d7c219f315525e329a4024f9'

for required in (SOURCE, ARCHIVE, ANNOTATIONS):
    assert required.is_file(), required
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == EXPECTED_DATA_SHA256
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
with zipfile.ZipFile(SOURCE) as source_zip:
    source_zip.extractall('/content')
if len(list((WORK_ROOT / 'images').glob('*.jpg'))) != 500:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as data_zip:
        data_zip.extractall(WORK_ROOT)
assert len(list((WORK_ROOT / 'images').glob('*.jpg'))) == 500

In [ ]:
%cd /content/Vietnamese-VQA-Explanation
%pip install -q -e '.[train]'

In [ ]:
OUTPUT = DRIVE_ROOT / 'predictions/M0_smoke.jsonl'
!python scripts/run_m0.py \
  --annotations "{ANNOTATIONS}" \
  --image-root "{WORK_ROOT}" \
  --output "{OUTPUT}" \
  --split validation --limit 20 --seed 20260918 --load-in-4bit

In [ ]:
import json

predictions = [json.loads(line) for line in OUTPUT.read_text(encoding='utf-8').splitlines()]
assert len(predictions) == 20, len(predictions)
valid = sum(item['format_valid'] for item in predictions)
print(f'Hoàn thành M0: {len(predictions)} mẫu; đúng định dạng: {valid}/{len(predictions)}')
for item in predictions[:3]:
    print('\n', item['question'])
    print('GT:', item['reference_answer'])
    print('M0:', item['prediction_answer'])
    print('Giải thích:', item['prediction_explanation'])